# A polytrope against the stellar kernel

A PROTOTYPE of the compute gate (Prompt 268, COMPUTE_GATE_II.md): it is not in the catalogue and not shipped. It reads its input as the text `gravitas_input` and its settings as the JSON text `gravitas_params`, and returns `gravitas_output`, a dict of numbers. Python only, numpy and scipy only.


In [ ]:
import base64, json, math
import numpy as np
from scipy.integrate import solve_ivp
doc = json.loads(gravitas_input)
assert doc.get("format") == "gravitas.grid" and doc.get("formatVersion") == 1, "this notebook reads gravitas.grid/1"
assert [a["name"] for a in doc["axes"]] == ["n"] and [o["name"] for o in doc["outputs"]] == ["xi1", "mass", "rhoRatio"], "the grid must have the axis n and the outputs xi1, mass, rhoRatio"
ns = [float(v) for v in doc["axes"][0]["values"]]
assert all(0 <= v < 5 for v in ns), "refused: the polytropic index must satisfy 0 <= n < 5 (n = 5 has an infinite radius, n > 5 an infinite mass, n < 0 is not a star)"
raw = np.frombuffer(base64.b64decode(doc["storage"]["data"]), dtype="<f8")
K = len(ns)
table = {"xi1": raw[:K], "mass": raw[K:2 * K], "rhoRatio": raw[2 * K:]}

In [ ]:
def lane_emden(n):
    x0 = 1e-6
    th0 = 1 - x0 ** 2 / 6 + n * x0 ** 4 / 120
    ph0 = -x0 / 3 + n * x0 ** 3 / 30
    def f(xi, y):
        return [y[1], -(1.0 if n == 0 else max(y[0], 0.0) ** n) - 2 * y[1] / xi]
    def hit(xi, y):
        return y[0]
    hit.terminal = True
    hit.direction = -1
    s = solve_ivp(f, (x0, 100), [th0, ph0], method="DOP853", rtol=1e-12, atol=1e-14, events=hit)
    xi1 = float(s.t_events[0][0])
    ph1 = float(s.y_events[0][0][1])
    return xi1, -xi1 ** 2 * ph1, xi1 / (-3 * ph1)

res = [lane_emden(v) for v in ns]
sol = {"xi1": [r[0] for r in res], "mass": [r[1] for r in res], "rhoRatio": [r[2] for r in res]}
rel = {k: [abs(a - b) / max(abs(a), abs(b)) for a, b in zip(sol[k], table[k].tolist())] for k in sol}
gravitas_output = {"answer": {"n": ns, "solution": sol, "relativeDifferenceFromTable": rel}}